In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q transformers==4.40.1
!pip install -q peft==0.10.0
!pip install -q sentence-transformers
!pip install -q accelerate


In [ ]:
# Example: extract your uploaded LoRA zip

!unzip -q /content/2_qwen1.5-1.8b-dpo-lora_reasoningupdated-20250603T023020Z-1-001.zip -d /content/lora_reasoning/

# Extract test set zip
!unzip -q /content/2_qwen1.5-1.8b-dpo-lora_updated.zip -d /content/testset/

!unzip -q /content/output_testset_wip.zip -d /content/testset/output_testset_wip/y


# Check test files
import glob
test_json_paths = glob.glob("/content/testset/output_testset_wip/**/*.json", recursive=True)
print(f"Found {len(test_json_paths)} test conversations.")


replace /content/lora_reasoning/2_qwen1.5-1.8b-dpo-lora_reasoningupdated/merges.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/lora_reasoning/2_qwen1.5-1.8b-dpo-lora_reasoningupdated/added_tokens.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/lora_reasoning/2_qwen1.5-1.8b-dpo-lora_reasoningupdated/chat_template.jinja? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/lora_reasoning/2_qwen1.5-1.8b-dpo-lora_reasoningupdated/special_tokens_map.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/lora_reasoning/2_qwen1.5-1.8b-dpo-lora_reasoningupdated/README.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/lora_reasoning/2_qwen1.5-1.8b-dpo-lora_reasoningupdated/tokenizer_config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/lora_reasoning/2_qwen1.5-1.8b-dpo-lora_reasoningupdated/training_args.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/lora_reasoning/2_qwen1.5-1.8b-dpo-lora_reasoningupdated/log_h

In [ ]:
!mkdir -p /content/offload

In [ ]:
import json
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base model — use GPU
base_model_name = "Qwen/Qwen1.5-1.8B"
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    offload_folder="/content/offload",
    trust_remote_code=True
)

In [ ]:

# Path to adapter config
adapter_config_path = "/content/lora_reasoning/2_qwen1.5-1.8b-dpo-lora_reasoningupdated/adapter_config.json"

# Load adapter config
with open(adapter_config_path, "r") as f:
    adapter_config = json.load(f)

# Remove unsupported fields (extensible list)
unsupported_keys = [
    "corda_config",
    "eva_config",
    "megatron_config",
    "exclude_modules",
    "auto_mapping",
    "alpha_pattern",
    "rank_pattern",
    "loftq_config",
    "layer_replication",
    "layers_pattern",
    "layers_to_transform",
    "trainable_token_indices",
    "modules_to_save",
    "revision",
    "lora_bias"
]

# Strip all problematic fields
for k in unsupported_keys:
    if k in adapter_config:
        print(f"Removing unsupported field: {k}")
        del adapter_config[k]

# Save the cleaned config
with open(adapter_config_path, "w") as f:
    json.dump(adapter_config, f, indent=2)

# Load the LoRA adapter — it will automatically go to GPU (since base_model is on device_map="auto")
model = PeftModel.from_pretrained(base_model, "/content/lora_reasoning/2_qwen1.5-1.8b-dpo-lora_reasoningupdated")
model.eval()




Removing unsupported field: corda_config
Removing unsupported field: eva_config
Removing unsupported field: megatron_config
Removing unsupported field: exclude_modules
Removing unsupported field: auto_mapping
Removing unsupported field: alpha_pattern
Removing unsupported field: rank_pattern
Removing unsupported field: loftq_config
Removing unsupported field: layer_replication
Removing unsupported field: layers_pattern
Removing unsupported field: layers_to_transform
Removing unsupported field: trainable_token_indices
Removing unsupported field: modules_to_save
Removing unsupported field: revision
Removing unsupported field: lora_bias


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): lora.Linear(
                (base_layer): Linear(in_features=2048

In [ ]:
# --- FINAL CLEAN INFERENCE LOOP BLOCK ---

import os
from tqdm import tqdm
import json
import torch

# Make output folder
os.makedirs("/content/qwen_outputs", exist_ok=True)

# Build prompt function
def build_prompt(conversation):
    prompt = ""
    for turn in conversation:
        if not isinstance(turn, dict):
            continue
        if turn.get("speaker") == "user":
            prompt += f"User: {turn.get('text', '')}\n"
        elif turn.get("speaker") == "agent":
            prompt += f"Agent: {turn.get('text', '')}\n"
    prompt += "Agent: "
    return prompt

# Main inference loop
for json_path in tqdm(test_json_paths):
    with open(json_path, "r") as f:
        data = json.load(f)

    # Handle Conversation
    conversation = data.get("Conversation", [])
    if not isinstance(conversation, list) or len(conversation) == 0:
        print(f"Skipping {json_path}: Conversation invalid or empty.")
        continue

    prompt = build_prompt(conversation)

    # Handle Contradiction
    contradiction = data.get("Contradiction", {})
    if not isinstance(contradiction, dict):
        curr_pref = ""
    else:
        curr_pref = contradiction.get("curr_pref", "")

    # Handle Response
    response = data.get("Response", {})
    if not isinstance(response, dict):
        preferred_response = ""
    else:
        preferred_response = response.get("preferred_response", "")

    # Tokenize and move to GPU
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract AgentAnswer
    if "Agent:" in generated_text:
        agent_reply = generated_text.split("Agent:")[-1].strip()
    else:
        agent_reply = generated_text.strip()

    # Save output
    out_data = {
        "ConversationID": os.path.basename(json_path),
        "AgentAnswer": agent_reply,
        "curr_pref": curr_pref,
        "preferred_response": preferred_response
    }

    out_path = os.path.join("/content/qwen_outputs", os.path.basename(json_path))
    with open(out_path, "w") as f_out:
        json.dump(out_data, f_out, indent=2)



 33%|███▎      | 30/90 [03:39<07:50,  7.84s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Skipping /content/testset/output_testset_wip/output_testset_wip/financialConsultation/conversation_financialConsultation_persona7_sample0.json: Conversation invalid or empty.


 51%|█████     | 46/90 [06:05<07:29, 10.22s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Skipping /content/testset/output_testset_wip/output_testset_wip/homeDecoration/conversation_homeDecoration_persona3_sample0.json: Conversation invalid or empty.


100%|██████████| 90/90 [11:32<00:00,  7.70s/it]


In [ ]:
# Zip outputs
!zip -r /content/qwen_outputs.zip /content/qwen_outputs

# Copy to Google Drive
!cp /content/qwen_outputs.zip /content/drive/MyDrive/qwen_outputs_reasoning.zip  # adjust name if needed


  adding: content/qwen_outputs/ (stored 0%)
  adding: content/qwen_outputs/conversation_homeDecoration_persona7_sample0.json (deflated 43%)
  adding: content/qwen_outputs/tokenizer.json (deflated 45%)
  adding: content/qwen_outputs/conversation_movieRecommendation_persona4_sample0.json (deflated 40%)
  adding: content/qwen_outputs/conversation_datingConsultation_persona7_sample0.json (deflated 47%)
  adding: content/qwen_outputs/conversation_studyConsultation_persona7_sample0.json (deflated 43%)
  adding: content/qwen_outputs/conversation_movieRecommendation_persona3_sample0.json (deflated 42%)
  adding: content/qwen_outputs/adapter_config.json (deflated 68%)
  adding: content/qwen_outputs/conversation_financialConsultation_persona8_sample0.json (deflated 43%)
  adding: content/qwen_outputs/conversation_healthConsultation_persona3_sample0.json (deflated 47%)
  adding: content/qwen_outputs/conversation_homeDecoration_persona4_sample0.json (deflated 42%)
  adding: content/qwen_outputs/co